# Phase 5: Train Model
Trains XGBoost Classifier with early stopping and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import pickle
import json

# Load processed data
df = pd.read_csv('../data/processed/train_sample_processed.csv')

y = df['isFraud']
X = df.drop(columns=['isFraud'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Fraud imbalance ratio
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric=['auc', 'aucpr'],
    early_stopping_rounds=50,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = model.predict(X_val)

print("ROC-AUC:", roc_auc_score(y_val, y_pred_proba))
print("PR-AUC:", average_precision_score(y_val, y_pred_proba))
print(classification_report(y_val, y_pred))

# Save model
with open('../model/fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save threshold
threshold_data = {"fraud_threshold": 0.72}
with open('../model/threshold.json', 'w') as f:
    json.dump(threshold_data, f)

# Save metadata
metadata = {
    "model_type": "XGBClassifier",
    "roc_auc": roc_auc_score(y_val, y_pred_proba),
    "pr_auc": average_precision_score(y_val, y_pred_proba)
}
with open('../model/metadata.json', 'w') as f:
    json.dump(metadata, f)
